# Module 3: Statistical Test

## Core Objective
To determine if the observed distribution of accessory modules within the defined slot space is statistically significant or merely a result of random chance, thereby identifying genuine insertion "hotspots" and "coldspots".

## Key Workflow & Features

1.  **Null Hypothesis (H₀)**:
    *   The test is based on a "Slot-Blind Insertion" model, which assumes that any accessory gene has an equal probability of occupying any available non-core position across the entire pangenome's flexible gene pool.

2.  **Permutation Testing**:
    *   The module performs thousands of random simulations (`--permutations`). In each simulation, it randomly re-distributes the total number of observed accessory genes into the available slot positions.
    *   This generates a null distribution of expected counts for each slot type.

3.  **Dual-Tier Analysis (Crucial Feature)**:
    *   The entire permutation test is conducted independently on two separate tiers of data to provide nuanced biological insights:
        *   **Tier A (All Accessory Genes)**: Tests for hotspots driven by the insertion of *any* accessory gene, whether single or part of a multi-gene cassette. This reveals broad patterns of plasticity.
        *   **Tier B (Multi-Gene Modules Only)**: A targeted test that only considers genes belonging to validated multi-gene cassettes. This is highly powerful for identifying hotspots governed by specific insertion mechanisms (e.g., site-specific integrases).

4.  **Statistical Inference & Reporting**:
    *   Compares the real observed counts to the simulated null distributions for each tier.
    *   Calculates `p-value_enrichment` (for hotspots), `p-value_depletion` (for coldspots), and `fold_enrichment` for every slot type.
    *   Generates two distinct output reports: one for general plasticity (`...slot_significance.csv`) and one for cassette-driven plasticity (`...multigene_module_significance.csv`).

In [ ]:
touch statistical_test/__init__.py

In [ ]:
# /GenoSynax_test/genosyntax/statistical_test/__init__.py

import os
import json

from genosyntax.utils.file_ops import load_json
from genosyntax.statistical_test.permutation_test import run_permutation_test
import pandas as pd

def run(args):
    """
    Entrypoint function for the 'run_stats' module.
    MODIFIED: Now handles and saves two separate output reports from the dual 
    permutation test.
    """
    if args.dry_run:
        print(f"--- Dry Run for 'run_stats' ---")
        print("Arguments received:", vars(args))
        if not os.path.exists(args.slot_analysis_dir):
            print(f"  [ERROR] Input path not found: {args.slot_analysis_dir}")
            return
        print("  All input paths are valid.")
        print("Dry run complete.")
        return
        
    output_dir = args.output_dir
    reports_dir = os.path.join(output_dir, "reports")
    os.makedirs(reports_dir, exist_ok=True)

    print("====== GenoSyntax: Running Module 'statistical_test' ======")
    
    # Load data from the 'slot_analysis' step
    slot_db_path = os.path.join(args.slot_analysis_dir, 'intermediate/slot_database.json')
    am_def_path = os.path.join(args.slot_analysis_dir, 'intermediate/accessory_module_definitions.json')
    
    print(f"[*] Loading slot database from: {slot_db_path}")
    slot_database = load_json(slot_db_path)
    
    print(f"[*] Loading accessory module definitions from: {am_def_path}")
    accessory_modules = load_json(am_def_path)
    
    if not slot_database or not accessory_modules:
        print("[FATAL] Could not load necessary input files. Please ensure the 'analyze_slots' module ran successfully.")
        return

    # --- MODIFIED: Run the dual test and receive two DataFrames ---
    results_df_all, results_df_multi = run_permutation_test(
        slot_database, accessory_modules, num_permutations=args.permutations
    )
    
    # --- MODIFIED: Save both output files with distinct, clear names ---
    
    # Save the original, primary report for all accessory modules
    if not results_df_all.empty:
        report_path_all = os.path.join(reports_dir, "permutation_test_slot_significance.csv")
        results_df_all.to_csv(report_path_all, index=False)
        print(f"\n[SUCCESS] Report for ALL accessory modules saved to: {report_path_all}")
    else:
        print("\n[INFO] No results were generated for the 'all modules' analysis.")

    # --- NEW: Save the new, additional report for multi-gene modules (cassettes) ---
    if not results_df_multi.empty:
        report_path_multi = os.path.join(reports_dir, "permutation_test_multigene_module_significance.csv")
        results_df_multi.to_csv(report_path_multi, index=False)
        print(f"[SUCCESS] Additional report for MULTI-GENE modules (cassettes) saved to: {report_path_multi}")
    else:
        print("[INFO] No results were generated for the 'multi-gene modules' analysis.")

    print(f"\n====== 'run_stats' Module Finished Successfully! ======")
    print(f"[*] Final reports saved to: {reports_dir}")

    return True

statistical_test/permutation_test.py

In [ ]:
# /GenoSynax_test/genosyntax/statistical_test/permutation_test.py

import numpy as np
import pandas as pd
from collections import Counter
from tqdm import tqdm
import random

def run_permutation_test(slot_database, accessory_modules, num_permutations=10000):
    """
    Performs a permutation test to identify significant hotspots and coldspots.
    VERSION 3 (MODIFIED): Includes a second, separate analysis specifically for 
    multi-gene modules to test for cassette-specific hotspots.
    """
    print("\n--- Task: Running Permutation Test for Slot Significance (v3 - Dual Analysis) ---")

    # ==============================================================================
    # --- ANALYSIS A: ALL ACCESSORY MODULES (>= 1 GENE) ---
    # This is the original, unchanged analysis.
    # ==============================================================================
    print("\n--- Analysis A: Testing ALL accessory modules (single- and multi-gene) ---")
    
    # --- 1A. Tally Observations and Prepare Data Pools ---
    print("  Step 1A: Tallying observations for all accessory genes...")
    
    all_module_member_sets = {g for m in accessory_modules for g in m['members']}
    
    observed_counts_per_slot_type = Counter()
    all_genes_in_slots = []
    slot_type_of_each_gene = []

    for genome_id, slots in slot_database.items():
        for slot_id, slot_info in slots.items():
            slot_type = slot_info['slot_id_low_res']
            for gene in slot_info['content']:
                all_genes_in_slots.append(gene)
                slot_type_of_each_gene.append(slot_type)
                if gene in all_module_member_sets:
                    observed_counts_per_slot_type[slot_type] += 1

    total_genes_in_slots = len(all_genes_in_slots)
    total_module_genes_in_slots = sum(1 for g in all_genes_in_slots if g in all_module_member_sets)
    unique_slot_types = sorted(list(set(slot_type_of_each_gene)))
    
    print(f"  Found {total_module_genes_in_slots} total accessory gene instances among {total_genes_in_slots} available positions.")
    
    report_df_all = pd.DataFrame() # Initialize in case of early exit
    if total_module_genes_in_slots > 0 and total_genes_in_slots > 0:
        # --- 2A. Run Permutations ---
        print(f"\n  Step 2A: Running {num_permutations} permutations for all accessory modules...")
        
        simulation_matrix = np.zeros((num_permutations, len(unique_slot_types)), dtype=int)
        slot_type_map = {st: i for i, st in enumerate(unique_slot_types)}
        all_indices = list(range(total_genes_in_slots))
        
        for i in tqdm(range(num_permutations), desc="  Simulating (All Modules)"):
            random_indices = random.sample(all_indices, total_module_genes_in_slots)
            simulated_slot_types = [slot_type_of_each_gene[j] for j in random_indices]
            sim_counts = Counter(simulated_slot_types)
            for slot_type, count in sim_counts.items():
                if slot_type in slot_type_map:
                    simulation_matrix[i, slot_type_map[slot_type]] = count
        
        # --- 3A. Calculate p-values ---
        print("\n  Step 3A: Calculating significance for all accessory modules...")
        results_all = []
        total_genes_per_slot_type = Counter(slot_type_of_each_gene)
        for i, slot_type in enumerate(unique_slot_types):
            observed_count = observed_counts_per_slot_type.get(slot_type, 0)
            simulation_counts = simulation_matrix[:, i]
            p_enrichment = (np.sum(simulation_counts >= observed_count) + 1) / (num_permutations + 1)
            p_depletion = (np.sum(simulation_counts <= observed_count) + 1) / (num_permutations + 1)
            mean_simulated = np.mean(simulation_counts)
            fold_enrichment = observed_count / mean_simulated if mean_simulated > 0 else float('inf')
            results_all.append({
                'slot_type': slot_type, 'observed_module_gene_count': observed_count,
                'mean_simulated_count': round(mean_simulated, 2), 'total_genes_in_slot': total_genes_per_slot_type[slot_type],
                'p_value_enrichment': round(p_enrichment, 5), 'p_value_depletion': round(p_depletion, 5),
                'fold_enrichment': round(fold_enrichment, 2) if fold_enrichment != float('inf') else 'inf'
            })
        report_df_all = pd.DataFrame(results_all).sort_values(by='p_value_enrichment', ascending=True)

    # ==============================================================================
    # --- NEW ANALYSIS B: MULTI-GENE MODULES ONLY (> 1 GENE) ---
    # This new section performs the specific test you requested.
    # ==============================================================================
    print("\n--- Analysis B: Testing MULTI-GENE modules (cassettes) ONLY ---")

    # --- 1B. Filter for multi-gene modules and re-tally observations ---
    print("  Step 1B: Tallying observations for multi-gene module members...")
    
    multi_gene_modules = [m for m in accessory_modules if m.get('num_genes', 0) > 1]
    multi_gene_member_sets = {g for m in multi_gene_modules for g in m['members']}

    observed_counts_per_slot_type_multi = Counter()
    for gene, slot_type in zip(all_genes_in_slots, slot_type_of_each_gene):
        if gene in multi_gene_member_sets:
            observed_counts_per_slot_type_multi[slot_type] += 1
    
    total_multi_gene_module_events = sum(observed_counts_per_slot_type_multi.values())
    
    print(f"  Found {total_multi_gene_module_events} multi-gene module instances among {total_genes_in_slots} available positions.")

    report_df_multi = pd.DataFrame() # Initialize in case of early exit
    if total_multi_gene_module_events > 0 and total_genes_in_slots > 0:
        # --- 2B. Run Permutations for multi-gene modules ---
        print(f"\n  Step 2B: Running {num_permutations} permutations for multi-gene modules...")
        
        # Reuse the same simulation setup, just change the number of events to sample
        simulation_matrix_multi = np.zeros((num_permutations, len(unique_slot_types)), dtype=int)
        
        for i in tqdm(range(num_permutations), desc="  Simulating (Multi-Gene)"):
            random_indices = random.sample(all_indices, total_multi_gene_module_events)
            simulated_slot_types = [slot_type_of_each_gene[j] for j in random_indices]
            sim_counts = Counter(simulated_slot_types)
            for slot_type, count in sim_counts.items():
                if slot_type in slot_type_map:
                    simulation_matrix_multi[i, slot_type_map[slot_type]] = count

        # --- 3B. Calculate p-values for multi-gene modules ---
        print("\n  Step 3B: Calculating significance for multi-gene modules...")
        results_multi = []
        for i, slot_type in enumerate(unique_slot_types):
            observed_count = observed_counts_per_slot_type_multi.get(slot_type, 0)
            simulation_counts = simulation_matrix_multi[:, i]
            p_enrichment = (np.sum(simulation_counts >= observed_count) + 1) / (num_permutations + 1)
            p_depletion = (np.sum(simulation_counts <= observed_count) + 1) / (num_permutations + 1)
            mean_simulated = np.mean(simulation_counts)
            fold_enrichment = observed_count / mean_simulated if mean_simulated > 0 else float('inf')
            results_multi.append({
                'slot_type': slot_type, 'observed_module_gene_count': observed_count,
                'mean_simulated_count': round(mean_simulated, 2), 'total_genes_in_slot': total_genes_per_slot_type[slot_type],
                'p_value_enrichment': round(p_enrichment, 5), 'p_value_depletion': round(p_depletion, 5),
                'fold_enrichment': round(fold_enrichment, 2) if fold_enrichment != float('inf') else 'inf'
            })
        report_df_multi = pd.DataFrame(results_multi).sort_values(by='p_value_enrichment', ascending=True)

    print("\nPermutation tests finished.")
    # --- MODIFIED: Return both DataFrames ---
    return report_df_all, report_df_multi

# Module 4: Mechanism Analysis

## Core Objective
To synthesize all prior findings and prepare high-quality data inputs for specialized, downstream analyses that explore the potential molecular and evolutionary mechanisms driving the observed genomic architecture.

## Key Sub-Modules & Features

1.  **Motif Discovery Preparation (`motif`)**:
    *   Identifies statistically significant hotspots using the dual reports from Module 3.
    *   For each hotspot type (general vs. cassette-specific), it extracts the non-genic DNA sequences flanking the slot boundaries across all relevant genomes.
    *   Generates two separate sets of FASTA files (`motif_analysis_all_modules/` and `motif_analysis_multigene_modules/`), ready for differential motif discovery tools like MEME-ChIP.

2.  **Evolutionary Dynamics Analysis (`evo_dynamics`)**:
    *   Calculates the **Protein Evolutionary Distance (PED)** for each accessory module, a measure of its sequence divergence (1 - average pairwise identity).
    *   Merges this data with the SPI report, allowing for direct analysis of the relationship between structural mobility (SPI) and evolutionary rate (PED).
    *   Performs a Fisher's Exact Test to identify slot types that are statistically enriched for hosting highly mobile (high-SPI) modules.

3.  **Adjacency Module Analysis (`adjacency_analysis`)**:
    *   Performs a deep functional analysis of all validated multi-gene (adjacency) modules.
    *   Categorizes modules based on the confidence of their functional annotations (e.g., "High-Confidence Functional Module").
    *   Generates a detailed narrative report, a summary CSV, and network files (nodes and edges) suitable for visualization in tools like Cytoscape.

4.  **Genome Layout Extraction (`extract_layouts`)**:
    *   Generates a simple, intuitive CSV file that maps the physical layout of anchors and slots for every high-quality genome.
    *   The output (`all_genome_layouts.csv`) contains the element type (Anchor/Slot), start coordinate, and end coordinate, making it ideal for creating genome map visualizations.

In [ ]:
# /GenoSynax_test/genosyntax/mechanism_analysis/__init__.py

import os
import pandas as pd
from collections import defaultdict

from genosyntax.utils.file_ops import load_json
from .motif_finder import extract_boundary_sequences
from .evolutionary_analyzer import run_evolutionary_analysis
from .adjacency_analyzer import run_adjacency_analysis
from .layout_extractor import run_layout_extraction
from .cds_extractor import run_cds_extraction

def run(args):
    """
    Entrypoint for 'explore_mechanisms' module.
    This version removes overly broad error handling to allow for proper debugging.
    """
    if args.dry_run:
        print(f"--- Dry Run for 'explore_mechanisms' ---")
        return True
        
    output_dir = args.output_dir
    os.makedirs(output_dir, exist_ok=True)

    print("\n====== GenoSyntax: Running Module 'explore_mechanisms' ======")
    
    print("--- Loading prerequisite data from previous modules ---")
    def_arch_dir = args.def_arch_dir
    slot_analysis_dir = args.slot_analysis_dir
    
    architecture_def = load_json(os.path.join(def_arch_dir, 'architecture_definition.json'))
    gene_order_db = load_json(os.path.join(def_arch_dir, 'intermediate/gene_order_database.json'))
    gene_coord_db = load_json(os.path.join(def_arch_dir, 'intermediate/gene_coordinate_database.json'))
    
    if not all([architecture_def, gene_order_db, gene_coord_db]):
        print("[FATAL] Could not load essential data from 'define_architecture' step. Aborting.")
        return False

    # --- Sub-module Dispatcher ---
    
    if 'extract_layouts' in args.analyses:
        print("\n--- Running Task: Extracting All Genome Layouts ---")
        run_layout_extraction(output_dir, architecture_def, gene_order_db, gene_coord_db)
    
    if 'extract_cds_layout' in args.analyses:
        print("\n--- Running Task: Extracting All CDS Layouts ---")
        gene_annotation_map = load_json(os.path.join(def_arch_dir, 'intermediate/gene_annotation_map.json'))
        if not gene_annotation_map:
            print("[FATAL] Could not load gene_annotation_map.json, required for CDS layout extraction. Aborting task.")
        else:
            run_cds_extraction(output_dir, architecture_def, gene_coord_db, gene_annotation_map)

    if 'evo_dynamics' in args.analyses:
        print("\n--- Running Task: Evolutionary Dynamics Analysis ---")
        df_spi = pd.read_csv(os.path.join(slot_analysis_dir, 'reports/module_dynamics_report.csv'))
        accessory_modules = load_json(os.path.join(slot_analysis_dir, 'intermediate/accessory_module_definitions.json'))
        run_evolutionary_analysis(
            output_dir, df_spi, accessory_modules, gene_coord_db, 
            args.pharokka_dir, getattr(args, 'spi_enrichment_threshold', 1.0)
        )

    if 'adjacency_analysis' in args.analyses:
        print("\n--- Running Task: Adjacency Module Functional Analysis ---")
        accessory_modules = load_json(os.path.join(slot_analysis_dir, 'intermediate/accessory_module_definitions.json'))
        gene_annotation_map = load_json(os.path.join(def_arch_dir, 'intermediate/gene_annotation_map.json'))
        run_adjacency_analysis(output_dir, accessory_modules, gene_annotation_map)

    if 'motif' in args.analyses:
        print("\n--- Running Task: Motif Discovery Preparation (All Modes) ---")
        slot_database = load_json(os.path.join(slot_analysis_dir, 'intermediate/slot_database.json'))
        
        # Mode 1: ALL accessory modules
        print("  -> Mode 1/3: Checking for 'all module' hotspots...")
        stats_report_all_path = os.path.join(args.stats_dir, 'reports/permutation_test_slot_significance.csv')
        if os.path.exists(stats_report_all_path):
            stats_report_all = pd.read_csv(stats_report_all_path)
            motif_dir_all = os.path.join(output_dir, "motif_analysis_all_modules")
            os.makedirs(motif_dir_all, exist_ok=True)
            extract_boundary_sequences(
                slot_database, gene_order_db, gene_coord_db, args.pharokka_dir, motif_dir_all,
                statistical_report_df=stats_report_all
            )
        else:
            print("     [INFO] Statistics report for 'all modules' not found. Skipping.")
        
        # (The rest of the motif logic remains the same, it is correct)
        print("  -> Mode 2/3: Checking for 'multi-gene module' hotspots...")
        stats_report_multi_path = os.path.join(args.stats_dir, 'reports/permutation_test_multigene_module_significance.csv')
        if os.path.exists(stats_report_multi_path):
            stats_report_multi = pd.read_csv(stats_report_multi_path)
            motif_dir_multi = os.path.join(output_dir, "motif_analysis_multigene_modules")
            os.makedirs(motif_dir_multi, exist_ok=True)
            extract_boundary_sequences(
                slot_database, gene_order_db, gene_coord_db, args.pharokka_dir, motif_dir_multi,
                statistical_report_df=stats_report_multi
            )
        else:
            print("     [INFO] Statistics report for 'multi-gene modules' not found. Skipping.")
        
        print("  -> Mode 3/3: Checking for slots enriched in 'high-SPI' modules...")
        high_spi_enrichment_path = os.path.join(output_dir, 'reports/high_spi_slot_enrichment.csv')
        if os.path.exists(high_spi_enrichment_path):
            high_spi_enrichment_df = pd.read_csv(high_spi_enrichment_path)
            significant_slots = high_spi_enrichment_df[high_spi_enrichment_df['q_value'] < 0.05]['slot_type'].tolist()
            if significant_slots:
                targeted_slot_instances = defaultdict(dict)
                for genome_id, slots in slot_database.items():
                    for slot_id, slot_info in slots.items():
                        if slot_info['slot_id_low_res'] in significant_slots:
                            targeted_slot_instances[genome_id][slot_id] = slot_info
                motif_dir_high_spi = os.path.join(output_dir, "motif_analysis_high_spi_enriched")
                os.makedirs(motif_dir_high_spi, exist_ok=True)
                extract_boundary_sequences(
                    slot_database, gene_order_db, gene_coord_db, args.pharokka_dir, motif_dir_high_spi,
                    target_slot_instances=targeted_slot_instances
                )
            else:
                print("     [INFO] No statistically significant slots for high-SPI modules found. Skipping.")
        else:
            print("     [INFO] High-SPI enrichment report not found. Skipping. (Hint: Run 'evo_dynamics' analysis).")
        
    print("\n====== 'explore_mechanisms' Module Finished Successfully! ======")
    return True

mechanism_analysis/layout_extractor.py

In [ ]:
# /GenoSynax_test/genosyntax/mechanism_analysis/layout_extractor.py

import json
import pandas as pd
from tqdm import tqdm
from collections import defaultdict
import os

def _get_anchor_permutation_map(architecture_def, gene_order_db, passed_genomes):
    """
    Creates a dictionary mapping each genome_id to its true anchor permutation.
    This logic mirrors the macro_analysis module for consistency.
    """
    anchor_skeletons = {a['anchor_id']: set(a['skeleton']) for a in architecture_def['anchor_definitions']}
    gene_to_anchor_map = defaultdict(list)
    for anchor_id, skeleton_set in anchor_skeletons.items():
        for gene in skeleton_set:
            gene_to_anchor_map[gene].append(anchor_id)

    permutation_map = {}
    for genome_id in passed_genomes:
        gene_order = gene_order_db.get(genome_id, [])
        if not gene_order: continue
            
        located_anchors = []
        found_anchors_in_genome = set()
        for i, gene in enumerate(gene_order):
            if gene in gene_to_anchor_map:
                for anchor_id in gene_to_anchor_map[gene]:
                    if anchor_id not in found_anchors_in_genome:
                        located_anchors.append({'pos': i, 'id': anchor_id})
                        found_anchors_in_genome.add(anchor_id)
        
        located_anchors.sort(key=lambda x: x['pos'])
        permutation_map[genome_id] = tuple(a['id'] for a in located_anchors)
        
    return permutation_map

def run_layout_extraction(output_dir, architecture_def, gene_order_db, gene_coord_db):
    """
    Main function to extract the true structural layouts of all high-quality genomes.
    """
    # ============================ START OF FIX ============================
    # Create the reports subdirectory if it doesn't exist. This is the fix.
    reports_dir = os.path.join(output_dir, "reports")
    os.makedirs(reports_dir, exist_ok=True)
    # ============================= END OF FIX =============================
    
    # Get essential metadata and data structures
    anchors = {a['anchor_id']: a for a in architecture_def['anchor_definitions']}
    passed_genomes = architecture_def['metadata']['qc_summary']['passed_genomes_list']

    print("  -> Determining true anchor permutations for all genomes...")
    genome_to_permutation_map = _get_anchor_permutation_map(architecture_def, gene_order_db, passed_genomes)

    layout_data = []
    print(f"  -> Processing {len(passed_genomes)} high-quality genomes to extract layouts...")

    for genome_id in tqdm(passed_genomes, desc="  Extracting Layouts", leave=False):
        if genome_id not in gene_coord_db or not gene_coord_db[genome_id]: continue
            
        genome_coords = gene_coord_db[genome_id]
        genome_length = max(c['end'] for c in genome_coords)
        known_permutation = genome_to_permutation_map.get(genome_id, [])
        
        located_anchors_with_coords = []
        for anchor_id in known_permutation:
            skeleton_genes = set(anchors[anchor_id]['skeleton'])
            coords_for_this_anchor = [c for c in genome_coords if c['group_id'] in skeleton_genes]
            if coords_for_this_anchor:
                start_bp = min(c['start'] for c in coords_for_this_anchor)
                end_bp = max(c['end'] for c in coords_for_this_anchor)
                located_anchors_with_coords.append({'id': anchor_id, 'start_bp': start_bp, 'end_bp': end_bp})
        
        last_pos = 1
        # Upstream Slot
        if located_anchors_with_coords:
            first_anchor_start = located_anchors_with_coords[0]['start_bp']
            if first_anchor_start > last_pos:
                layout_data.append({'genome_id': genome_id, 'element_id': 'Slot_Upstream', 'element_type': 'Slot', 'start': last_pos, 'end': first_anchor_start - 1})
        
        # Anchors and Inter-Anchor Slots
        for i, anchor in enumerate(located_anchors_with_coords):
            layout_data.append({'genome_id': genome_id, 'element_id': anchor['id'], 'element_type': 'Anchor', 'start': anchor['start_bp'], 'end': anchor['end_bp']})
            # This logic needed a small correction to handle inter-anchor slots correctly
            if i < len(located_anchors_with_coords) - 1:
                current_anchor_end = anchor['end_bp']
                next_anchor_start = located_anchors_with_coords[i+1]['start_bp']
                
                if next_anchor_start > current_anchor_end + 1:
                    next_anchor_id = located_anchors_with_coords[i+1]['id']
                    slot_id = f"Slot({anchor['id']}->{next_anchor_id})"
                    layout_data.append({'genome_id': genome_id, 'element_id': slot_id, 'element_type': 'Slot', 'start': current_anchor_end + 1, 'end': next_anchor_start - 1})

        # Downstream Slot
        last_anchor_end = located_anchors_with_coords[-1]['end_bp'] if located_anchors_with_coords else 0
        if last_anchor_end < genome_length:
            layout_data.append({'genome_id': genome_id, 'element_id': 'Slot_Downstream', 'element_type': 'Slot', 'start': last_anchor_end + 1, 'end': genome_length})
            
    df_layout = pd.DataFrame(layout_data)
    if not df_layout.empty:
        df_layout['length'] = df_layout['end'] - df_layout['start'] + 1
    
    output_path = os.path.join(reports_dir, "all_genome_layouts.csv")
    df_layout.to_csv(output_path, index=False)
    print(f"  -> Genome layout data for {df_layout['genome_id'].nunique()} genomes saved to: {output_path}")

mechanism_analysis/cds_extractor.py

In [ ]:
# FILE: genosyntax/mechanism_analysis/cds_extractor.py (NEW FILE)

import os
import json
import pandas as pd
from tqdm import tqdm

# =============================================================================
# --- 1. FUNCTIONAL CLASSIFICATION LOGIC ---
# This is a direct copy of the logic used inside GenoSyntax's functional_annotator.py
# to ensure consistency in categorization.
# =============================================================================

FUNCTIONAL_MAP = {
    "Structural & Infection": [
        'tail', 'baseplate', 'capsid', 'head', 'portal', 'terminase', 
        'lysis', 'holin', 'endolysin', 'sheath', 'fiber', 'tape measure',
        'structural', 'virion'
    ],
    "Replication & Regulation": [
        'dna polymerase', 'helicase', 'primase', 'ligase', 'replication',
        'rna polymerase', 'transcription', 'sigma factor', 'regulator',
        'repressor', 'dna binding'
    ],
    "Genome Maintenance & Metabolism": [
        'recombination', 'exonuclease', 'endonuclease', 'repair', 
        'nucleotide', 'synthesis', 'kinase', 'synthase', 'reductase',
        'metabolism', 'folate', 'integrase', 'recombinase', 'transposase'
    ],
    "Hypothetical & Unknown": [
        'hypothetical', 'unknown', 'uncharacterized', 'predicted protein'
    ]
}

def _get_function_category(product_description: str) -> str:
    """
    Assigns a high-level functional category to a gene based on its product description.
    """
    if not isinstance(product_description, str):
        return "Hypothetical & Unknown"
        
    desc_lower = product_description.lower()
    
    for category, keywords in FUNCTIONAL_MAP.items():
        if any(keyword in desc_lower for keyword in keywords):
            return category
            
    # If no keywords match, default to a general category
    return "Other"

# =============================================================================
# --- 2. MAIN SCRIPT LOGIC (REFACTORED) ---
# =============================================================================

def run_cds_extraction(output_dir, arch_def, gene_coord_db, gene_annotation_map):
    """
    Main function to load GenoSyntax outputs and generate a comprehensive CDS table.
    """
    reports_dir = os.path.join(output_dir, "reports")
    os.makedirs(reports_dir, exist_ok=True)
    
    # --- Step 1: Extract the list of high-quality genomes ---
    passed_genomes = arch_def.get('metadata', {}).get('qc_summary', {}).get('passed_genomes_list', [])
    if not passed_genomes:
        print("  [INFO] No high-quality genomes found in the architecture definition file. Skipping CDS layout extraction.")
        return
    
    print(f"  -> Found {len(passed_genomes)} high-quality genomes to process.")

    # --- Step 2: Iterate through genomes and genes to build the table ---
    all_cds_data = []
    
    for genome_id in tqdm(passed_genomes, desc="  Extracting CDS", leave=False):
        gene_list = gene_coord_db.get(genome_id)
        if not gene_list:
            continue

        for gene_info in gene_list:
            group_id = gene_info.get('group_id')
            if not group_id:
                continue
                
            product = gene_annotation_map.get(group_id, "Unknown product")
            category = _get_function_category(product)
            
            all_cds_data.append({
                'genome_id': genome_id,
                'start': gene_info.get('start'),
                'end': gene_info.get('end'),
                'strand': gene_info.get('strand'),
                'product': product,
                'function_category': category,
                'gene_family_id': group_id # Add gene family ID for extra utility
            })
    
    # --- Step 3: Convert to DataFrame and save to CSV ---
    if not all_cds_data:
        print("  [INFO] No CDS data was extracted. The output file will not be created.")
        return

    df = pd.DataFrame(all_cds_data)
    
    # Ensure a useful column order
    df = df[['genome_id', 'start', 'end', 'strand', 'gene_family_id', 'product', 'function_category']]
    
    output_path = os.path.join(reports_dir, "all_genomes_cds_layout.csv")
    df.to_csv(output_path, index=False)
    
    print(f"  -> Comprehensive CDS table for {df['genome_id'].nunique()} genomes saved to: {output_path}")

mechanism_analysis/evolutionary_analyzer.py

In [ ]:
# /GenoSynax_test/genosyntax/mechanism_analysis/evolutionary_analyzer.py

import os
import json
import subprocess
from io import StringIO
import pandas as pd
from Bio import SeqIO, AlignIO
from tqdm import tqdm
from itertools import combinations
from collections import Counter
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests

# --- Helper Function for Protein Distance ---
def _calculate_pairwise_identity(alignment):
    """Calculates the average pairwise identity from a Bio.AlignIO alignment object."""
    num_seqs = len(alignment)
    if num_seqs < 2: return None
    
    total_identity, num_pairs = 0, 0
    for i, j in combinations(range(num_seqs), 2):
        seq1, seq2 = str(alignment[i].seq), str(alignment[j].seq)
        matches, length = 0, 0
        for char1, char2 in zip(seq1, seq2):
            if char1 != '-' or char2 != '-':
                length += 1
                if char1 == char2:
                    matches += 1
        if length > 0:
            total_identity += (matches / length)
            num_pairs += 1
            
    return total_identity / num_pairs if num_pairs > 0 else None

# --- Core Analysis Functions ---
def _calculate_protein_evolutionary_distance(accessory_modules, gene_coord_db, protein_seqs, temp_dir):
    """
    Calculates Protein Evolutionary Distance (PED) for each accessory module.
    PED = 1 - Average Pairwise Identity.
    """
    print("  -> Sub-task: Calculating Protein Evolutionary Distance (PED)...")
    results = []
    
    for module in tqdm(accessory_modules, desc="  Calculating PED"):
        module_id, member_genes = module['module_id'], module['members']
        
        all_family_records = []
        for gene_family in member_genes:
            family_records_for_gene = []
            for genome_id, gene_list in gene_coord_db.items():
                gene_info = next((g for g in gene_list if g.get('group_id') == gene_family), None)
                if gene_info and gene_info.get('locus_tag'):
                    locus_tag = gene_info['locus_tag']
                    if genome_id in protein_seqs and locus_tag in protein_seqs[genome_id]:
                        family_records_for_gene.append(protein_seqs[genome_id][locus_tag])
            all_family_records.append(family_records_for_gene)

        module_distances = []
        for i, records in enumerate(all_family_records):
            if len(records) < 2: continue
            
            gene_family = member_genes[i]
            unaligned_path = os.path.join(temp_dir, f"{gene_family}.faa")
            SeqIO.write(records, unaligned_path, "fasta")
            
            try:
                mafft_process = subprocess.run(
                    ['mafft', '--auto', '--quiet', unaligned_path], 
                    capture_output=True, text=True, check=True, encoding='utf-8'
                )
                if not mafft_process.stdout: continue
                alignment = AlignIO.read(StringIO(mafft_process.stdout), "fasta")
                avg_identity = _calculate_pairwise_identity(alignment)
                if avg_identity is not None:
                    module_distances.append(1 - avg_identity)
            except (subprocess.CalledProcessError, ValueError, FileNotFoundError) as e:
                if isinstance(e, FileNotFoundError):
                    print("\n[FATAL ERROR] MAFFT command not found. Is it installed and in your PATH?")
                    exit(1)
                continue
        
        if module_distances:
            avg_module_ped = sum(module_distances) / len(module_distances)
            results.append({'module_id': module_id, 'protein_evolutionary_distance': avg_module_ped})

    return pd.DataFrame(results)

def _analyze_high_spi_slot_enrichment(df_spi, spi_threshold):
    """
    Performs Fisher's Exact Test to find if high-SPI modules are enriched in certain slot types.
    """
    print(f"  -> Sub-task: Analyzing slot enrichment for high-SPI (>{spi_threshold}) modules...")
    df = df_spi.copy()
    df = df[df['observed_slots'] != 'None'].copy()
    df['spi_group'] = df['spi'].apply(lambda x: 'high' if x > spi_threshold else 'low')
    
    high_spi_slots, low_spi_slots = [], []
    for _, row in df.iterrows():
        slots = [s.strip() for s in row['observed_slots'].split(';')]
        if row['spi_group'] == 'high':
            high_spi_slots.extend(slots)
        else:
            low_spi_slots.extend(slots)

    high_counts, low_counts = Counter(high_spi_slots), Counter(low_spi_slots)
    all_slots = sorted(list(set(high_counts.keys()) | set(low_counts.keys())))
    
    enrichment_results = []
    total_high_obs, total_low_obs = sum(high_counts.values()), sum(low_counts.values())
    
    if total_high_obs == 0:
        print("    [INFO] No high-SPI modules found. Skipping enrichment analysis.")
        return pd.DataFrame()

    for slot in all_slots:
        a = high_counts.get(slot, 0)
        b = total_high_obs - a
        c = low_counts.get(slot, 0)
        d = total_low_obs - c
        
        odds_ratio, p_value = fisher_exact([[a, b], [c, d]], alternative='greater')
        enrichment_results.append({
            'slot_type': slot, 'high_spi_count_in_slot': a, 'low_spi_count_in_slot': c,
            'odds_ratio': odds_ratio, 'p_value': p_value
        })
        
    df_enrichment = pd.DataFrame(enrichment_results)
    if not df_enrichment.empty:
        _, q_values, _, _ = multipletests(df_enrichment['p_value'], alpha=0.05, method='fdr_bh')
        df_enrichment['q_value'] = q_values
        df_enrichment = df_enrichment.sort_values('p_value').reset_index(drop=True)

    return df_enrichment

# --- Main Entrypoint Function ---
def run_evolutionary_analysis(output_dir, df_spi, accessory_modules, gene_coord_db, pharokka_dir, spi_threshold):
    """
    Executes all evolutionary dynamics analyses.
    """
    # Setup directories
    reports_dir = os.path.join(output_dir, "reports")
    temp_dir = os.path.join(output_dir, "intermediate", "mafft_temp")
    os.makedirs(reports_dir, exist_ok=True)
    os.makedirs(temp_dir, exist_ok=True)

    # Pre-load all protein sequences for efficiency
    print("  -> Pre-loading protein sequences from Pharokka .faa files...")
    protein_seqs = {}
    genome_dirs = [d for d in os.listdir(pharokka_dir) if os.path.isdir(os.path.join(pharokka_dir, d))]
    for genome_id in tqdm(genome_dirs, desc="  Loading proteins"):
        faa_path = os.path.join(pharokka_dir, genome_id, "phanotate.faa")
        if os.path.exists(faa_path):
            try:
                protein_seqs[genome_id] = SeqIO.to_dict(SeqIO.parse(faa_path, "fasta"))
            except Exception:
                continue

    # Run analyses
    df_prot_dist = _calculate_protein_evolutionary_distance(accessory_modules, gene_coord_db, protein_seqs, temp_dir)
    df_slot_enrichment = _analyze_high_spi_slot_enrichment(df_spi, spi_threshold)

    # Merge and save final reports
    df_master = pd.merge(df_spi, df_prot_dist, on='module_id', how='left')
    
    master_path = os.path.join(reports_dir, "spi_vs_ped_analysis.csv")
    enrichment_path = os.path.join(reports_dir, "high_spi_slot_enrichment.csv")
    
    df_master.to_csv(master_path, index=False)
    if not df_slot_enrichment.empty:
        df_slot_enrichment.to_csv(enrichment_path, index=False)

    print(f"  -> Evolutionary dynamics report saved to: {master_path}")
    if not df_slot_enrichment.empty:
        print(f"  -> High-SPI slot enrichment report saved to: {enrichment_path}")

mechanism_analysis/adjacency_analyzer.py

In [ ]:
# /GenoSynax_test/genosyntax/mechanism_analysis/adjacency_analyzer.py

import os
import pandas as pd
from collections import defaultdict

def run_adjacency_analysis(output_dir, accessory_modules, gene_annotation_map):
    """
    Performs a deep functional analysis of adjacency (multi-gene) modules.
    Generates a detailed narrative report, a CSV summary, heatmap data, and network files.
    This version is hardened against empty inputs and ensures correct output.
    """
    analysis_dir = os.path.join(output_dir, "adjacency_module_analysis")
    os.makedirs(analysis_dir, exist_ok=True)
    
    adjacency_modules = [m for m in accessory_modules if m.get('module_type') == 'multi-gene']
    print(f"  -> Found {len(adjacency_modules)} adjacency (multi-gene) modules to analyze.")
    
    if not adjacency_modules:
        print("  [INFO] No adjacency modules were found in the input. Analysis complete.")
        # Create empty files to show the task ran
        open(os.path.join(analysis_dir, "adjacency_modules_narrative_report.txt"), 'w').write("No adjacency modules found.\n")
        pd.DataFrame().to_csv(os.path.join(analysis_dir, "adjacency_modules_summary.csv"), index=False)
        return

    report_sections = defaultdict(list)
    node_data, edge_data, heatmap_data, summary_data = [], [], [], []
    non_informative_keywords = ["hypothetical protein", "unknown function"]

    for module in sorted(adjacency_modules, key=lambda x: x.get('module_id', '')):
        module_id = module.get('module_id', 'Unknown_ID')
        members = module.get('members', [])
        num_genes = len(members)
        
        informative_annot_count = 0
        gene_details = [] 

        for i, gene_id in enumerate(members):
            annotation = gene_annotation_map.get(gene_id, "unknown function")
            is_informative = not any(keyword in annotation.lower() for keyword in non_informative_keywords)
            if is_informative:
                informative_annot_count += 1
            
            gene_details.append({'id': gene_id, 'annotation': annotation})
            heatmap_data.append({'module_id': module_id, 'gene_position': i, 'gene_id': gene_id, 'function_description': annotation})
            node_data.append({'NodeID': gene_id, 'ModuleID': module_id, 'FunctionDescription': annotation})

        if informative_annot_count == num_genes and num_genes > 0:
            category, confidence = "Category A: High-Confidence Functional Modules", "High"
        elif informative_annot_count > 0:
            category, confidence = "Category B: Plausible Functional Modules", "Medium"
        else:
            category, confidence = "Category C: Uncharacterized Modules", "Low"

        narrative = f"### Module: {module_id} ({num_genes} genes)\n"
        narrative += f"Annotation Confidence: {confidence} ({informative_annot_count}/{num_genes} informative)\n\n"
        narrative += "| Gene ID    | Function Description (from Panaroo)\n"
        narrative += "|------------|--------------------------------------------------\n"
        for gene in gene_details:
            narrative += f"| {gene['id']:<10} | {gene['annotation']}\n"
        
        report_sections[category].append(narrative)

        summary_data.append({'module_id': module_id, 'num_genes': num_genes, 'confidence_category': category, 'informative_gene_count': informative_annot_count, 'member_ids': ", ".join(members)})

        if len(members) > 1:
            for i in range(len(members) - 1):
                edge_data.append({'Source': members[i], 'Target': members[i+1], 'InteractionType': 'PhysicalAdjacency', 'ModuleID': module_id})

    final_report = "====== Adjacency Module Narrative Functional Report ======\n\n"
    final_report += "This report categorizes multi-gene adjacency modules based on the confidence of their functional annotations, sourced from Panaroo.\n\n"
    category_order = ["Category A: High-Confidence Functional Modules", "Category B: Plausible Functional Modules", "Category C: Uncharacterized Modules"]

    for cat_name in category_order:
        if cat_name in report_sections:
            final_report += f"===========================================================\n"
            final_report += f"## {cat_name}\n"
            final_report += f"===========================================================\n\n"
            separator = "\n---\n\n"
            final_report += separator.join(report_sections[cat_name])
            final_report += "\n\n"

    with open(os.path.join(analysis_dir, "adjacency_modules_narrative_report.txt"), 'w') as f: f.write(final_report)
    print(f"  -> Detailed narrative report saved.")
    
    pd.DataFrame(summary_data).to_csv(os.path.join(analysis_dir, "adjacency_modules_summary.csv"), index=False)
    print(f"  -> CSV summary report saved.")
    
    pd.DataFrame(heatmap_data).to_csv(os.path.join(analysis_dir, "adjacency_module_heatmap_data.csv"), index=False)
    print(f"  -> Heatmap data saved.")
    
    pd.DataFrame(node_data).to_csv(os.path.join(analysis_dir, "adjacency_nodes.csv"), index=False)
    print(f"  -> Cytoscape node table saved.")
    
    pd.DataFrame(edge_data).to_csv(os.path.join(analysis_dir, "adjacency_edges.csv"), index=False)
    print(f"  -> Cytoscape edge table saved.")
    print(f"  -> All outputs are in: {analysis_dir}")

mechanism_analysis/motif_finder.py

In [ ]:
# /GenoSynax_test/genosyntax/mechanism_analysis/motif_finder.py

import os
import json
from tqdm import tqdm
from Bio import SeqIO
import pandas as pd
from collections import defaultdict

def _sanitize_for_header(text: str) -> str:
    """
    Sanitizes a string to be a single, safe word for a FASTA header.
    """
    text = text.replace('(', '').replace(')', '')
    text = text.replace(' -> ', '-to-')
    text = text.replace(' ', '-')
    text = text.replace(':', '-')
    return text

# --- NEW: Helper function for final FASTA sanitization ---
def _sanitize_fasta_for_meme(file_path: str, min_length: int):
    """
    Reads a FASTA file, filters out sequences shorter than min_length,
    and overwrites the file with linearized, clean FASTA records.
    This makes the file robustly compatible with MEME-ChIP.
    """
    # Check if the file was created and is not empty
    if not os.path.exists(file_path) or os.path.getsize(file_path) == 0:
        return # Do nothing if the file doesn't exist or is empty

    try:
        # Read all records, keeping only those that meet the length requirement
        kept_records = []
        for record in SeqIO.parse(file_path, "fasta"):
            if len(record.seq) >= min_length:
                kept_records.append(record)

        # Overwrite the original file with the filtered and sanitized records
        # SeqIO.write handles the linearization automatically.
        with open(file_path, 'w') as f:
            SeqIO.write(kept_records, f, "fasta")

        if kept_records:
             print(f"  - Sanitized {os.path.basename(file_path)}: Kept {len(kept_records)} sequences >= {min_length}bp.")

    except Exception as e:
        print(f"  [ERROR] Could not sanitize file {file_path}: {e}")


def extract_boundary_sequences(
    slot_database,
    gene_order_db,
    gene_coord_db,
    pharokka_output_dir,
    output_dir,
    window_size=200,
    p_value_threshold=0.05,
    min_meme_seq_length=20, # NEW: Parameter for MEME sanitization
    statistical_report_df=None,
    target_slot_instances=None
):
    """
    Extracts boundary sequences from genomic slots.
    VERSION 6: Adds a final sanitization step to prepare files for MEME-ChIP,
    including sequence linearization and length filtering.
    """
    
    # --- Steps 1, 2, 3, 4 remain the same as the previous version ---
    
    # --- Determine Operational Mode ---
    if target_slot_instances is not None:
        mode = "Targeted"
        database_to_process = target_slot_instances
        print(f"\n--- Task: Extracting Boundary Sequences (Targeted Mode, Window: {window_size}bp) ---")
    elif statistical_report_df is not None and not statistical_report_df.empty:
        mode = "Hotspot/Coldspot"
        database_to_process = slot_database
        print(f"\n--- Task: Extracting Boundary Sequences (Hotspot/Coldspot Mode, Window: {window_size}bp) ---")
        print("  Step 1: Classifying slot types...")
        hot_slots = set(statistical_report_df[statistical_report_df['p_value_enrichment'] < p_value_threshold]['slot_type'])
        cold_slots = set(statistical_report_df[statistical_report_df['p_value_depletion'] < p_value_threshold]['slot_type'])
    else:
        print("\n[ERROR] No valid input provided for motif extraction.")
        return

    print("  Step 2: Loading genome sequences...")
    all_possible_genomes = list(slot_database.keys())
    nucleotide_seqs = {}
    for genome_id in tqdm(all_possible_genomes, desc="  Loading NT sequences"):
        fna_path = next((os.path.join(pharokka_output_dir, genome_id, f) for f in os.listdir(os.path.join(pharokka_output_dir, genome_id)) if f.endswith(('.fna', '.fasta', 'fa')) and 'reoriented' in f), None)
        if not fna_path:
            fna_path = next((os.path.join(pharokka_output_dir, genome_id, f) for f in os.listdir(os.path.join(pharokka_output_dir, genome_id)) if f.endswith(('.fna', '.fasta', 'fa'))), None)
        if fna_path and os.path.exists(fna_path):
            try:
                records = list(SeqIO.parse(fna_path, "fasta"))
                if len(records) == 1: nucleotide_seqs[genome_id] = records[0]
            except Exception: # Ignore parsing errors for corrupted files
                pass

    print("\n  Step 3: Extracting boundary sequences...")
    output_seqs = defaultdict(list)
    
    for genome_id, slots in tqdm(database_to_process.items(), desc="  Processing genomes"):
        if genome_id not in nucleotide_seqs: continue
        full_gene_order = gene_order_db.get(genome_id, [])
        coords_by_group = {g['group_id']: g for g in reversed(gene_coord_db.get(genome_id, []))}
        genome_sequence = str(nucleotide_seqs[genome_id].seq)
        
        for slot_id, slot_info in slots.items():
            if mode == "Hotspot/Coldspot":
                if slot_info['slot_id_low_res'] in hot_slots: slot_class = "hot"
                elif slot_info['slot_id_low_res'] in cold_slots: slot_class = "cold"
                else: slot_class = "neutral"
            else:
                slot_class = "high_spi_freq"

            if not slot_info['content']: continue
            try:
                order_map = {gene: i for i, gene in enumerate(full_gene_order)}
                first_gene_global_idx = order_map[slot_info['content'][0]]
                last_gene_global_idx = first_gene_global_idx + len(slot_info['content']) - 1
            except (KeyError, ValueError):
                continue

            backend_gene_id = full_gene_order[first_gene_global_idx - 1] if first_gene_global_idx > 0 else None
            frontend_gene_id = full_gene_order[last_gene_global_idx + 1] if last_gene_global_idx < len(full_gene_order) - 1 else None
            sanitized_slot_id = _sanitize_for_header(slot_info['slot_id_low_res'])
            
            if backend_gene_id and coords_by_group.get(backend_gene_id):
                backend_coord = coords_by_group[backend_gene_id]
                first_gene_coord = coords_by_group.get(slot_info['content'][0])
                if first_gene_coord:
                    extract_start = backend_coord['end']
                    limit = first_gene_coord['start'] - 1
                    extract_end = min(extract_start + window_size, limit)
                    if extract_end > extract_start:
                        seq = genome_sequence[extract_start:extract_end]
                        header = f">{genome_id}__{sanitized_slot_id}__backend__abs_start={extract_start}"
                        output_seqs[f"{slot_class}_backend"].append(f"{header}\n{seq}")

            if frontend_gene_id and coords_by_group.get(frontend_gene_id):
                frontend_coord = coords_by_group[frontend_gene_id]
                last_gene_coord = coords_by_group.get(slot_info['content'][-1])
                if last_gene_coord:
                    extract_end = frontend_coord['start'] - 1
                    limit = last_gene_coord['end']
                    extract_start = max(extract_end - window_size, limit)
                    if extract_end > extract_start:
                        seq = genome_sequence[extract_start:extract_end]
                        header = f">{genome_id}__{sanitized_slot_id}__frontend__abs_start={extract_start}"
                        output_seqs[f"{slot_class}_frontend"].append(f"{header}\n{seq}")

    print("\n  Step 4: Writing initial output FASTA files...")
    created_files = []
    for category, sequences in output_seqs.items():
        if sequences:
            file_path = os.path.join(output_dir, f"{category}_boundaries.fasta")
            with open(file_path, 'w') as f:
                f.write("\n".join(sequences))
            print(f"  - Wrote {len(sequences)} sequences to {os.path.basename(file_path)}")
            created_files.append(file_path)

    # --- NEW: STEP 5 ---
    # Final sanitization pass to ensure MEME-ChIP compatibility
    print(f"\n  Step 5: Sanitizing FASTA files for MEME-ChIP (min length: {min_meme_seq_length}bp)...")
    if not created_files:
        print("  - No files were created, skipping sanitization.")
    else:
        for f_path in created_files:
            _sanitize_fasta_for_meme(f_path, min_length=min_meme_seq_length)

    print("\n--- Motif Finder data preparation finished ---")